# U Net: Arquitectura de segmentación semantica

In [ ]:
"""
Materia: Inteligencia Artificial
Tema: Segmentación de peces con U-Net (binaria pez/fondo)
"""
import os, random, json                      # utilidades del sistema y semilla
from pathlib import Path                     # rutas
import cv2                                   # procesamiento de imágenes
import numpy as np                           # arreglos
import pandas as pd                          # tablas (mask_qc.csv)
import matplotlib.pyplot as plt              # figuras
import tensorflow as tf                      # framework
from tensorflow import keras                 # API de alto nivel
from tensorflow.keras import layers          # capas

from google.colab import drive               # montaje de Drive
drive.mount('/content/drive')


SEED = 42
IMG_SIZE = 256
ROOT = Path('/content/drive/MyDrive/sylfish')
PROCESSED_DIR = ROOT / 'data' / 'processed'  # igual que en el 01
REPORTS_DIR = ROOT / 'reports'
BATCH_SIZE = 16
UMBRAL_MASCARA = 0.5                         # se ajustará sobre val en la sección G

os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

print('TF', tf.__version__, '| GPUs:', tf.config.list_physical_devices('GPU'))

Ahora vamos a cargar las cachés de la U-Net y quedarnos solo con las imágenes cuya máscara pasó el control de calidad (qc_ok). Las máscaras vienen en $\{0, 255\}$; las dividimos entre 255 y añadimos el canal para obtener la forma $(256, 256, 1)$.

In [ ]:
qc = pd.read_csv(PROCESSED_DIR / 'mask_qc.csv')
validas = set(qc.loc[qc['qc_ok'], 'filepath'])    # rutas con máscara aprobada

def cargar_split(split):
    d = np.load(PROCESSED_DIR / f'unet_256_{split}.npz', allow_pickle=True)
    X, M, fp = d['X'], d['M'], d['filepath']
    ok = np.array([f in validas for f in fp])      # filtro qc_ok
    X, M, fp = X[ok], M[ok], fp[ok]
    M = (M.astype('float32') / 255.0)[..., None]   # {0,1}, forma (N,256,256,1)
    return X, M, fp

X_train, M_train, fp_train = cargar_split('train')
X_val,   M_val,   fp_val   = cargar_split('val')
X_test,  M_test,  fp_test  = cargar_split('test')

for nombre, X, M in [('train', X_train, M_train), ('val', X_val, M_val), ('test', X_test, M_test)]:
    print(nombre, X.shape, X.dtype, M.shape, np.unique(M), f'pez medio={M.mean():.3f}')

In [ ]:
perm = np.random.RandomState(SEED).permutation(len(X_train))   # orden aleatorio reproducible
X_train, M_train, fp_train = X_train[perm], M_train[perm], fp_train[perm]   # reordenamos juntos imagen, máscara y ruta
print(fp_train[:5])

Ahora vamos a armar el pipeline tf.data. Normalizamos las imágenes a $[0,1]$ y, solo en train, barajamos y aplicamos aumento: volteos y rotaciones de 90° iguales para imagen y máscara, y brillo y contraste solo para la imagen.

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE                       # paralelismo automático

def preparar(x, m):
    x = tf.cast(x, tf.float32) / 255.0            # imagen en [0,1]
    m = tf.cast(m, tf.float32)                    # máscara en {0,1}
    return x, m

def aumentar(x, m):
    z = tf.concat([x, m], axis=-1)                # (256,256,4): misma transformación para ambas
    z = tf.image.random_flip_left_right(z)
    z = tf.image.random_flip_up_down(z)
    z = tf.image.rot90(z, k=tf.random.uniform([], 0, 4, dtype=tf.int32))   # 0, 90, 180 o 270 grados
    x, m = z[..., :3], z[..., 3:]                 # separamos de nuevo
    x = tf.image.random_brightness(x, 0.15)       # fotométrico: solo la imagen
    x = tf.image.random_contrast(x, 0.8, 1.2)
    x = tf.clip_by_value(x, 0.0, 1.0)
    return x, m

def construir_ds(X, M, entrenar):
    ds = tf.data.Dataset.from_tensor_slices((X, M.astype(np.uint8)))      # uint8 para ahorrar memoria
    if entrenar:
        ds = ds.shuffle(2048, seed=SEED, reshuffle_each_iteration=True)
    ds = ds.map(preparar, num_parallel_calls=AUTOTUNE)
    if entrenar:
        ds = ds.map(aumentar, num_parallel_calls=AUTOTUNE)                # solo en train
    return ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)

ds_train = construir_ds(X_train, M_train, entrenar=True)
ds_val   = construir_ds(X_val,   M_val,   entrenar=False)
ds_test  = construir_ds(X_test,  M_test,  entrenar=False)

xb, mb = next(iter(ds_train))
print(xb.shape, xb.dtype, mb.shape, mb.dtype, 'valores de M:', np.unique(mb.numpy()))

Ahora vamos a comprobar visualmente que el aumento mueve imagen y máscara juntas: el contorno rojo debe seguir al pez en todos los casos.

In [ ]:
fig, ejes = plt.subplots(2, 4, figsize=(16, 8))
xb, mb = next(iter(ds_train))
for ax, x, m in zip(ejes.ravel(), xb[:8].numpy(), mb[:8].numpy()):
    vis = (x * 255).astype(np.uint8).copy()
    cont, _ = cv2.findContours(m[..., 0].astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    cv2.drawContours(vis, cont, -1, (255, 0, 0), 2)    # rojo = máscara aumentada
    ax.imshow(vis); ax.axis('off')
plt.tight_layout(); plt.show()

# Arquitectura U-Net

Ahora vamos a montar la U-Net capa por capa. La entrada es una imagen de $256 \times 256 \times 3$ y la salida es un mapa de $256 \times 256 \times 1$ con la probabilidad de que cada píxel sea pez.

Encoder. Cuatro niveles; en cada uno aplicamos dos convoluciones $3 \times 3$ con BatchNormalization y ReLU, guardamos la salida para la skip connection y reducimos con MaxPool $2 \times 2$.

Decoder. Cuatro niveles simétricos: una convolución transpuesta $2 \times 2$ con strides=2, que duplica la resolución y reduce los canales a los del nivel, concatenación con la salida guardada del encoder en la misma resolución y dos convoluciones $3 \times 3$ con BatchNormalization y ReLU.

Salida. Una convolución $1 \times 1$ con 1 filtro y activación sigmoide:

$$\hat{y}_p = \sigma(z_p) = \frac{1}{1 + e^{-z_p}} \in (0, 1)$$

Un píxel $p$ se clasifica como pez si $\hat{y}_p > \tau$, con $\tau = 0.5$ por defecto (se ajustará en val).

In [ ]:
# fijamos una semilla
tf.random.set_seed(SEED)

# fijamos tamaño de las imagenes
img_size = (256,256,3)

entrada = layers.Input(shape=img_size)

# ahora pasamos a la primera parte de la arquitectura, el encoder, que reduce las dimensiones y extrae caracteristicas semanticas de aprendizaje

# Encoder - nivel 1
e1 = layers.Conv2D(16, (3, 3), padding='same')(entrada)
e1 = layers.BatchNormalization()(e1)
e1 = layers.Activation('relu')(e1)
e1 = layers.Conv2D(16, (3, 3), padding='same')(e1)
e1 = layers.BatchNormalization()(e1)
e1 = layers.Activation('relu')(e1)
skip_connection1 = e1 # guardamos la salida del nivel 1 en una skip connection que luego vamos a concatenar
e1 = layers.MaxPooling2D((2, 2))(e1)

# Encoder - nivel 2

e2 = layers.Conv2D(32, (3, 3), padding='same')(e1)
e2 = layers.BatchNormalization()(e2)
e2 = layers.Activation('relu')(e2)
e2 = layers.Conv2D(32, (3, 3), padding='same')(e2)
e2 = layers.BatchNormalization()(e2)
e2 = layers.Activation('relu')(e2)
skip_connection2 = e2
e2 = layers.MaxPooling2D((2, 2))(e2)

# Encoder - nivel 3

e3 = layers.Conv2D(64, (3, 3), padding='same')(e2)
e3 = layers.BatchNormalization()(e3)
e3 = layers.Activation('relu')(e3)
e3 = layers.Conv2D(64, (3, 3), padding='same')(e3)
e3 = layers.BatchNormalization()(e3)
e3 = layers.Activation('relu')(e3)
skip_connection3 = e3
e3 = layers.MaxPooling2D((2, 2))(e3)

# Encoder - nivel 4

e4 = layers.Conv2D(128, (3, 3), padding='same')(e3)
e4 = layers.BatchNormalization()(e4)
e4 = layers.Activation('relu')(e4)
e4 = layers.Conv2D(128, (3, 3), padding='same')(e4)
e4 = layers.BatchNormalization()(e4)
e4 = layers.Activation('relu')(e4)
skip_connection4 = e4
e4 = layers.MaxPooling2D((2, 2))(e4)

# Bottleneck, intermediario entre el encoder y el decoder

b = layers.Conv2D(256, (3, 3), padding='same')(e4)
b = layers.BatchNormalization()(b)
b = layers.Activation('relu')(b)
b = layers.Conv2D(256, (3, 3), padding='same')(b)
b = layers.BatchNormalization()(b)
b = layers.Activation('relu')(b)


# ahora pasamos a la ultima parte de la arquitectura, el decoder, el cual recupera las dimensiones de la imagen orginal y junto con las skip connections genera el mapa final de caracteristicas semanticas, con clases

# Decoder - nivel 4

d4 = layers.Conv2DTranspose(128, (2, 2), strides=2, padding='same')(b)
d4 = layers.Concatenate()([d4, skip_connection4])   # concatenamos con la skip conection del mismo nivel en el encoder, esto hace que las features se dupliquen, por lo que haremos convolucion para mezclarlas
d4 = layers.Conv2D(128, (3, 3), padding='same')(d4)
d4 = layers.BatchNormalization()(d4)
d4 = layers.Activation('relu')(d4)
d4 = layers.Conv2D(128, (3, 3), padding='same')(d4) # la segunda convolucion es para generar las features del nivel
d4 = layers.BatchNormalization()(d4)
d4 = layers.Activation('relu')(d4)

# Decoder - nivel 3

d3 = layers.Conv2DTranspose(64, (2, 2), strides=2, padding='same')(d4)
d3 = layers.Concatenate()([d3, skip_connection3])
d3 = layers.Conv2D(64, (3, 3), padding='same')(d3)
d3 = layers.BatchNormalization()(d3)
d3 = layers.Activation('relu')(d3)
d3 = layers.Conv2D(64, (3, 3), padding='same')(d3)
d3 = layers.BatchNormalization()(d3)
d3 = layers.Activation('relu')(d3)

# Decoder - nivel 2

d2 = layers.Conv2DTranspose(32, (2,2), strides = 2, padding='same')(d3)
d2 = layers.Concatenate()([d2, skip_connection2])
d2 = layers.Conv2D(32, (3, 3), padding='same')(d2)
d2 = layers.BatchNormalization()(d2)
d2 = layers.Activation('relu')(d2)
d2 = layers.Conv2D(32, (3, 3), padding='same')(d2)
d2 = layers.BatchNormalization()(d2)
d2 = layers.Activation('relu')(d2)

# Decoder - nivel 1

d1 = layers.Conv2DTranspose(16, (2,2), strides = 2, padding='same')(d2)
d1 = layers.Concatenate()([d1, skip_connection1])
d1 = layers.Conv2D(16, (3, 3), padding='same')(d1)
d1 = layers.BatchNormalization()(d1)
d1 = layers.Activation('relu')(d1)
d1 = layers.Conv2D(16, (3, 3), padding='same')(d1)
d1 = layers.BatchNormalization()(d1)
d1 = layers.Activation('relu')(d1)

# Luego generamos la salida a través de una convolución con un kernel 1x1, la cual le asigna a cada pixel una clase, como son solo 2, en este caso usaremos sigmoide como activación, con un umbral de clasificacion tao

salida = layers.Conv2D(1, (1, 1), activation='sigmoid')(d1)

# Ahora generamos nuestro modelo
U_net = keras.Model(inputs=entrada, outputs=salida)
U_net.summary()

## Pérdida y métricas

Ahora vamos a definir la función de pérdida y las métricas. Como el pez ocupa ≈ 15 % de los píxeles, combinamos dos términos: la entropía cruzada binaria, que penaliza cada píxel, y la pérdida Dice, que mide el solapamiento y no se deja engañar por el desbalance.

Con $p_i$ la probabilidad predicha, $y_i$ la máscara real y $N$ los píxeles de una imagen:

$$\text{Dice} = \frac{2\sum_i p_i\, y_i + \varepsilon}{\sum_i p_i + \sum_i y_i + \varepsilon}, \qquad \mathcal{L} = \text{BCE} + \left(1 - \text{Dice}\right)$$

La métrica principal es el IoU por imagen, promediado sobre las imágenes (mIoU), con la predicción binarizada en $\tau = 0.5$:

$$\text{IoU} = \frac{|P \cap Y|}{|P \cup Y|}$$

In [ ]:
EJES = [1, 2, 3]                                  # alto, ancho y canal: una cifra por imagen

def dice_coef(y_true, y_pred, eps=1e-6):
    inter = tf.reduce_sum(y_true * y_pred, axis=EJES)                              # solapamiento suave
    suma = tf.reduce_sum(y_true, axis=EJES) + tf.reduce_sum(y_pred, axis=EJES)
    return tf.reduce_mean((2.0 * inter + eps) / (suma + eps))                      # promedio sobre el lote

def bce_dice(y_true, y_pred):
    bce = tf.reduce_mean(keras.losses.binary_crossentropy(y_true, y_pred))         # término por píxel
    return bce + (1.0 - dice_coef(y_true, y_pred))                                 # más el término de solapamiento

def iou_por_imagen(y_true, y_pred, eps=1e-6):
    p = tf.cast(y_pred > UMBRAL_MASCARA, tf.float32)                               # binarizamos con tau
    inter = tf.reduce_sum(y_true * p, axis=EJES)
    union = tf.reduce_sum(y_true, axis=EJES) + tf.reduce_sum(p, axis=EJES) - inter
    return tf.reduce_mean((inter + eps) / (union + eps))                           # mIoU del lote

Compilamos el modelo

In [ ]:
U_net.compile(optimizer='adam', loss=bce_dice, metrics=[iou_por_imagen])

Probamos consistencia de la perdida y la metrica

In [ ]:
xb, mb = next(iter(ds_val))                       # un lote de val
print('perfecta  :', float(iou_por_imagen(mb, mb)), float(bce_dice(mb, tf.clip_by_value(mb, 1e-3, 1 - 1e-3))))
print('todo fondo:', float(iou_por_imagen(mb, tf.zeros_like(mb))))

## Entrenamiento de U-Net

Ahora vamos a entrenar la U-Net. Guardamos en Drive el mejor modelo según el mIoU de validación, reducimos la tasa de aprendizaje cuando la métrica se estanca y detenemos el entrenamiento si deja de mejorar. El historial se escribe a Drive en cada época para no perderlo si Colab se desconecta.

In [ ]:
MODELS_DIR = ROOT / 'models'                      # pesos en Drive
TABLES_DIR = REPORTS_DIR / 'tables'
MODELS_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR.mkdir(parents=True, exist_ok=True)

callbacks = [
    keras.callbacks.ModelCheckpoint(MODELS_DIR / 'unet_v1.keras',             # guarda solo el mejor
                                    monitor='val_iou_por_imagen', mode='max', save_best_only=True),
    keras.callbacks.ReduceLROnPlateau(monitor='val_iou_por_imagen', mode='max',
                                      factor=0.5, patience=3, min_lr=1e-5, verbose=1),
    keras.callbacks.EarlyStopping(monitor='val_iou_por_imagen', mode='max',
                                  patience=8, restore_best_weights=True),
    keras.callbacks.CSVLogger(TABLES_DIR / '03_unet_historial.csv'),          # una fila por época, en Drive
]

historial = U_net.fit(ds_train, validation_data=ds_val, epochs=40, callbacks=callbacks)

Como paró por early stopping, veamos algunas metricas

In [ ]:
val_iou = historial.history['val_iou_por_imagen']
mejor = int(np.argmax(val_iou))                                    # índice de la mejor época
print('épocas ejecutadas   :', len(val_iou))
print('mejor época         :', mejor + 1, '| val_iou:', round(val_iou[mejor], 4))
print('épocas sin mejorar  :', len(val_iou) - (mejor + 1))
print(U_net.evaluate(ds_val, verbose=0))                           # pérdida, iou y dice con los pesos restaurados

Ahora vamos a graficar cómo evolucionaron la pérdida, el IoU y la tasa de aprendizaje a lo largo de las épocas, marcando la mejor época según el mIoU de validación.

In [ ]:
h = historial.history                                    # métricas por época
epocas = np.arange(1, len(h['loss']) + 1)
mejor = int(np.argmax(h['val_iou_por_imagen'])) + 1      # época con mejor mIoU de val
FIGURES_DIR = REPORTS_DIR / 'figures'

fig, ejes = plt.subplots(1, 3, figsize=(18, 4.5))
ejes[0].plot(epocas, h['loss'], label='train'); ejes[0].plot(epocas, h['val_loss'], label='val')
ejes[0].set_title('Pérdida (BCE + Dice)'); ejes[0].set_ylabel('pérdida')
ejes[1].plot(epocas, h['iou_por_imagen'], label='train'); ejes[1].plot(epocas, h['val_iou_por_imagen'], label='val')
ejes[1].set_title('IoU por imagen'); ejes[1].set_ylabel('mIoU')
lr = h.get('learning_rate', h.get('lr'))                 # el nombre depende de la versión de Keras
if lr is not None:
    ejes[2].plot(epocas, lr, color='tab:green', label='tasa de aprendizaje')
    ejes[2].set_yscale('log'); ejes[2].set_title('Tasa de aprendizaje'); ejes[2].set_ylabel('lr')
for eje in ejes:
    eje.axvline(mejor, color='red', ls='--', alpha=0.6, label=f'mejor época ({mejor})')
    eje.set_xlabel('época'); eje.grid(alpha=0.3); eje.legend()
fig.tight_layout(); fig.savefig(FIGURES_DIR / '03_unet_curvas_entrenamiento.png', dpi=200); plt.show()

 ## Evaluación en validación

Ahora vamos a evaluar la U-Net sobre val con cifras exactas, no con el promedio por lotes de Keras. Primero predecimos todas las probabilidades, luego ajustamos el umbral $\tau$ y finalmente analizamos dónde falla.

In [ ]:
FIGURES_DIR = REPORTS_DIR / 'figures'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

P_val = U_net.predict(ds_val, verbose=0)[..., 0]        # (N,256,256) probabilidades; ds_val no se baraja, el orden es el de fp_val
Y_val = M_val[..., 0].astype(bool)                      # máscara real como booleano
print(P_val.shape, Y_val.shape, len(fp_val))            # los tres N deben coincidir (1066)

Ahora vamos a elegir el umbral $\tau$ sobre val. Para cada valor calculamos el IoU de cada imagen y lo promediamos (mIoU), y nos quedamos con el $\tau$ que lo maximiza.

In [ ]:
def iou_img(P, Y, tau, eps=1e-6):
    B = P > tau                                          # binarizamos con el umbral
    inter = (B & Y).sum(axis=(1, 2))                     # intersección por imagen
    union = (B | Y).sum(axis=(1, 2))                     # unión por imagen
    return (inter + eps) / (union + eps)

taus = np.round(np.arange(0.10, 0.91, 0.05), 2)
mious = []
for t in taus:
    mious.append(iou_img(P_val, Y_val, t).mean())        # mIoU por imagen para cada umbral
mious = np.array(mious)

TAU = float(taus[np.argmax(mious)])
print('mIoU en tau=0.5 :', round(float(mious[np.argmin(np.abs(taus - 0.5))]), 4))
print('mejor tau       :', TAU, '| mIoU:', round(float(mious.max()), 4))

fig, eje = plt.subplots(figsize=(6, 3.5))
eje.plot(taus, mious, marker='o'); eje.axvline(TAU, color='red', ls='--', label=f'tau = {TAU}')
eje.set_xlabel('umbral'); eje.set_ylabel('mIoU por imagen (val)'); eje.legend()
fig.tight_layout(); fig.savefig(FIGURES_DIR / '03_unet_umbral.png', dpi=200); plt.show()

Ahora vamos a ver la matriz de confusión a nivel de píxel con el umbral elegido y las métricas que se derivan de ella.

In [ ]:
import seaborn as sns                                    # mapa de calor

TP = int((B & Y_val).sum()); FP = int((B & ~Y_val).sum())
FN = int((~B & Y_val).sum()); TN = int((~B & ~Y_val).sum())

conf = np.array([[TN, FP], [FN, TP]])                    # filas = clase real, columnas = clase predicha
conf_pct = conf / conf.sum(axis=1, keepdims=True)        # normalizada por fila: proporción de cada clase real

etiquetas = np.empty((2, 2), dtype=object)               # texto de cada celda: conteo y porcentaje
for i in range(2):
    for j in range(2):
        etiquetas[i, j] = f'{conf[i, j] / 1e6:.2f} M\n{conf_pct[i, j]:.1%}'

nombres = ['Fondo', 'Pez']
fig, eje = plt.subplots(figsize=(6, 5))
sns.heatmap(conf_pct, annot=etiquetas, fmt='', cmap='Blues', vmin=0, vmax=1,
            xticklabels=nombres, yticklabels=nombres,
            linewidths=2, linecolor='white', annot_kws={'size': 14},
            cbar_kws={'label': 'proporción de la clase real'}, ax=eje)
eje.set_title(rf'Matriz de confusión de píxeles - val ($\tau$ = {TAU})')
eje.set_xlabel('Clase predicha'); eje.set_ylabel('Clase real')
fig.tight_layout(); fig.savefig(FIGURES_DIR / '03_unet_confusion_pixeles.png', dpi=200); plt.show()

pd.DataFrame(conf, index=['real fondo', 'real pez'], columns=['pred fondo', 'pred pez']).to_csv(TABLES_DIR / '03_unet_confusion_pixeles.csv')
print('precisión :', round(TP / (TP + FP), 4))
print('recall    :', round(TP / (TP + FN), 4))
print('Dice      :', round(2 * TP / (2 * TP + FP + FN), 4))
print('exactitud :', round((TP + TN) / conf.sum(), 4), '(poco informativa: ~85 % es fondo)')

Ahora vamos a desglosar el IoU por clase y a contar cuántas imágenes quedan por debajo de dos umbrales de calidad.

In [ ]:
iou_i = iou_img(P_val, Y_val, TAU)                       # IoU de cada imagen de val
clase = np.array([f.split('/')[0] for f in fp_val])
tabla = pd.DataFrame({'clase': clase, 'iou': iou_i}).groupby('clase')['iou'].agg(['count', 'mean', 'median', 'min']).round(4)
print(tabla)
print('IoU < 0.5:', int((iou_i < 0.5).sum()), '| IoU < 0.7:', int((iou_i < 0.7).sum()), 'de', len(iou_i))
tabla.to_csv(TABLES_DIR / '03_unet_iou_por_clase.csv')

Ahora vamos a mirar los 8 peores casos con un mapa de error: blanco es acierto, rojo es falso positivo (la red marca pez donde no hay) y azul es falso negativo (la red no marca pez donde sí hay).

In [ ]:
peores = np.argsort(iou_i)[:8]                           # los 8 IoU más bajos
fig, ejes = plt.subplots(2, 8, figsize=(24, 6.5))
for k, i in enumerate(peores):
    mapa = np.zeros((256, 256, 3), np.uint8)
    mapa[B[i] & Y_val[i]] = (255, 255, 255)              # TP
    mapa[B[i] & ~Y_val[i]] = (255, 0, 0)                 # FP
    mapa[~B[i] & Y_val[i]] = (0, 120, 255)               # FN
    ejes[0, k].imshow(X_val[i]); ejes[0, k].set_title(f'{Path(fp_val[i]).name}\nIoU={iou_i[i]:.2f}', fontsize=8); ejes[0, k].axis('off')
    ejes[1, k].imshow(mapa); ejes[1, k].axis('off')
fig.tight_layout(); fig.savefig(FIGURES_DIR / '03_unet_peores_val.png', dpi=200); plt.show()

with open(MODELS_DIR / 'unet_v1_umbral.json', 'w') as f:   # umbral fijado para test y la demo
    json.dump({'tau': TAU, 'miou_val': float(iou_i.mean())}, f, indent=2)

Ahora vamos a ver la otra cara de la evaluación: la distribución del IoU en val, los 8 mejores casos y 8 imágenes al azar. Los mejores muestran dónde red y etiqueta coinciden; la muestra al azar muestra el caso típico.

In [ ]:
def mostrar_casos(indices, archivo):
    fig, ejes = plt.subplots(2, len(indices), figsize=(3 * len(indices), 6.5))
    for k, i in enumerate(indices):
        mapa = np.zeros((256, 256, 3), np.uint8)
        mapa[B[i] & Y_val[i]] = (255, 255, 255)           # acierto
        mapa[B[i] & ~Y_val[i]] = (255, 0, 0)              # la red marca pez y la etiqueta no
        mapa[~B[i] & Y_val[i]] = (0, 120, 255)            # la etiqueta marca pez y la red no
        ejes[0, k].imshow(X_val[i]); ejes[0, k].set_title(f'{Path(fp_val[i]).name}\nIoU={iou_i[i]:.2f}', fontsize=8); ejes[0, k].axis('off')
        ejes[1, k].imshow(mapa); ejes[1, k].axis('off')
    fig.tight_layout(); fig.savefig(FIGURES_DIR / archivo, dpi=200); plt.show()

# distribución del IoU por imagen
print('percentiles 5, 25, 50, 75, 95:', np.round(np.percentile(iou_i, [5, 25, 50, 75, 95]), 3))
fig, eje = plt.subplots(figsize=(6, 3.5))
eje.hist(iou_i, bins=40, color='steelblue')
eje.set_xlabel('IoU por imagen (val)'); eje.set_ylabel('imágenes')
fig.tight_layout(); fig.savefig(FIGURES_DIR / '03_unet_hist_iou_val.png', dpi=200); plt.show()

mejores = np.argsort(iou_i)[::-1][:8]                      # los 8 IoU más altos
azar = np.random.RandomState(SEED).choice(len(iou_i), 8, replace=False)   # 8 al azar, reproducibles
mostrar_casos(mejores, '03_unet_mejores_val.png')
mostrar_casos(azar, '03_unet_azar_val.png')

**Conclusión:**
 La U-Net segmenta el pez con un acuerdo alto con las pseudo-máscaras en la mayoría de las imágenes de val: la mediana del IoU por imagen es $0.915$ (cuartiles $0.796$ y $0.951$), y en una muestra aleatoria de 8 imágenes, 7 tienen IoU $\geq 0.87$ con diferencias de uno o dos píxeles en el borde. El mIoU ($\approx 0.83$) queda por debajo de la mediana por una cola de imágenes de bajo IoU (P5 $= 0.411$).

En los 8 casos de menor IoU, a ojo, la red segmenta correctamente el pez en 7 de ellos y el desacuerdo se debe a que la pseudo-máscara de GrabCut marca otra región (suelo, borde del plato o lona); solo en koi_300 la red también falla. Algo parecido ocurre en telapia_690 (IoU $0.46$) de la muestra aleatoria. Esto sugiere que, al menos en parte, la cola de bajo IoU proviene de etiquetas erróneas y no de la red, por lo que el mIoU sería una cota inferior de la calidad real de la segmentación.

Estas observaciones son cualitativas: se basan en muestras pequeñas (8 imágenes por figura), los mejores casos son todos de una misma clase con fondo uniforme, y no se contrastaron con una referencia anotada a mano, por lo que no se cuantifica qué proporción de la cola corresponde a etiqueta errónea y cuál a error de la red. Además, la red reproduce el contorno suavizado de GrabCut (sin aletas finas), de modo que el detalle del borde queda limitado por la etiqueta.

Como mejora futura proponemos un autoentrenamiento: usar esta U-Net para generar máscaras sobre las imágenes descartadas por el control de calidad y reentrenar con ellas, comparando ambas versiones en val y revisando manualmente las pseudo-etiquetas para no reforzar los errores del modelo.

# Evaluación en test

Ahora vamos a evaluar una sola vez el modelo congelado sobre test, con el umbral fijado en val. No se ajusta nada después de ver estos resultados. Cargamos el mejor modelo guardado en Drive, comprobamos que reproduce la cifra de val y luego predecimos test.

In [ ]:
modelo_final = keras.models.load_model(MODELS_DIR / 'unet_v1.keras', compile=False)   # modelo congelado
with open(MODELS_DIR / 'unet_v1_umbral.json') as f:
    TAU = json.load(f)['tau']                                                          # umbral fijado en val

# comprobación: el modelo cargado debe reproducir el mIoU de val (≈ 0.833)
P_chk = modelo_final.predict(ds_val, verbose=0)[..., 0]
print('mIoU val (modelo cargado):', round(float(iou_img(P_chk, Y_val, TAU).mean()), 4), '| tau =', TAU)
del P_chk

In [ ]:
P_test = modelo_final.predict(ds_test, verbose=0)[..., 0]     # ds_test no se baraja: orden de fp_test
Y_test = M_test[..., 0].astype(bool)
B_t = P_test > TAU
iou_t = iou_img(P_test, Y_test, TAU)                          # IoU por imagen

print('mIoU test :', round(float(iou_t.mean()), 4))
print('mediana   :', round(float(np.median(iou_t)), 4), '| percentiles 5, 25, 75, 95:', np.round(np.percentile(iou_t, [5, 25, 75, 95]), 3))
print('IoU < 0.5 :', int((iou_t < 0.5).sum()), '| IoU < 0.7:', int((iou_t < 0.7).sum()), 'de', len(iou_t))

TPt = int((B_t & Y_test).sum()); FPt = int((B_t & ~Y_test).sum())
FNt = int((~B_t & Y_test).sum()); TNt = int((~B_t & ~Y_test).sum())
print('precisión:', round(TPt / (TPt + FPt), 4), '| recall:', round(TPt / (TPt + FNt), 4), '| Dice global:', round(2 * TPt / (2 * TPt + FPt + FNt), 4))

clase_t = np.array([f.split('/')[0] for f in fp_test])
tabla_t = pd.DataFrame({'clase': clase_t, 'iou': iou_t}).groupby('clase')['iou'].agg(['count', 'mean', 'median', 'min']).round(4)
print(tabla_t)

tabla_t.to_csv(TABLES_DIR / '03_unet_test_iou_por_clase.csv')
with open(TABLES_DIR / '03_unet_test_resumen.json', 'w') as f:
    json.dump({'tau': TAU, 'miou_test': float(iou_t.mean()), 'mediana': float(np.median(iou_t)),
               'iou_menor_05': int((iou_t < 0.5).sum()), 'n': int(len(iou_t))}, f, indent=2)

La evaluación con el Test Split fue consistente con los datos de validación, esto es otra prueba de que no hubo overfitting. En conclusión el modelo obtuvo un entrenamiento acorde a las mascaras, si existe algún tipo de error es por el error en las mismas, aunque no alcanzó el IoU esperado del 95 porciento, podriamos decir que es un buen modelo, mas no excelente.